In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.9/527.9 kB 18.4 MB/s eta 0:00:00


In [2]:
#praw imports
import praw
import pandas as pd
import os
import csv
import requests
import glob
import shutil

#GCP vision imports
from typing import List, Tuple
from PIL import Image
import statistics
import re
import numpy as np
from google.cloud import vision
from google.oauth2 import service_account

In [3]:
reddit = praw.Reddit(
    client_id = "client_id here",
    client_secret = "client_secret here",
    user_agent = "your app name"
)


In [4]:
#for gcp vision api
# Auth setup
credentials = service_account.Credentials.from_service_account_file("/content/chat-image-detector-b43386af8be8.json")
client = vision.ImageAnnotatorClient(credentials=credentials)

# Scams Which Will Be Searched On Reddit

In [5]:
scams = [

    {
        "keyword": "Work From Home",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "Scams"
    },
    {
        "keyword": "Work From Home",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "whatsapp"
    },
    {
        "keyword": "WFH",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "IsThisAScamIndia"
    },
    {
        "keyword": "Work From Home",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "scambait"
    },
     {
        "keyword": "Loan Scam",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "IsThisAScamIndia"
    },
     {
        "keyword": "Loan Scam",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "CreditCardsIndia"
    },
     {
        "keyword": "Wrong Number",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "Scams"
    },
     {
        "keyword": "Wrong Number",
        #"mandatory_keywords": ["scam","fraud", "remote work", "work from home","conversation"],
        "subreddit": "scambait"
    }

]

In [6]:


# Function to save images from a Reddit post
def save_post_images(post, image_urls,scam_dir):
    post_dir = os.path.join(scam_dir, post.id)
    os.makedirs(post_dir, exist_ok=True)

    for idx, url in enumerate(image_urls, start=1):
        try:
            response = requests.get(url)
            if response.status_code == 200:
                ext = url.split(".")[-1].split("?")[0]  # Get image extension safely
                filename = f"image{idx}.{ext}"
                filepath = os.path.join(post_dir, filename)
                with open(filepath, "wb") as f:
                    f.write(response.content)
        except Exception as e:
            print(f"Failed to download {url}: {e}")

# Fetching Images From Reddit

In [7]:
base_dir = "dataset_test"
os.makedirs(base_dir, exist_ok=True)
csv_file = os.path.join(base_dir, "metadata.csv")

# Create CSV file with headers once
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["post_id", "title", "url"])


for scam in scams:
    keyword = scam["keyword"]
    #mandatory_keywords = scam["mandatory_keywords"]
    subreddit_name = scam["subreddit"]

    print(f"\n🔍 Searching for: {keyword} in r/{subreddit_name}")

    # Create scam-specific folder

    scam_folder_name = keyword.replace(" ", "_") + "_scam"
    scam_dir = os.path.join(base_dir, scam_folder_name)
    os.makedirs(scam_dir, exist_ok=True)



    count = 0
    appended_post_count = 0

    subreddit = reddit.subreddit(subreddit_name)


        # ---------------- Search Posts ----------------
    for post in subreddit.search(keyword, sort="relevance", limit=50):
        images = []
        count +=1
        title_lower = post.title.lower()


        #Check single image
        if post.url.lower().endswith((".jpg", ".jpeg", ".png", ".gif")):
            images.append(post.url)
            save_post_images(post, [post.url],scam_dir)

        # Check gallery images
        if hasattr(post, 'is_gallery') and post.is_gallery:
            image_urls = []
            for item in post.gallery_data['items']:
                media_id = item['media_id']
                img_url = post.media_metadata[media_id]['s']['u'].replace("&amp;", "&")
                images.append(img_url)
            save_post_images(post, images,scam_dir)

        # Append to CSV only if at least one image is found
        if images:  # function returns True equivalent

            appended_post_count += 1
            with open(csv_file, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)  # quote all fields
                writer.writerow([post.id, post.title, post.url])

            print(f"Post {count} added with {len(images)} image(s).")
        else:  # function returns False equivalent
            print(f"Skipping Post : {count} | No image found")
            continue


    print("Total post's appended: ", appended_post_count)

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.




🔍 Searching for: Work From Home in r/Scams


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Skipping Post : 1 | No image found
Skipping Post : 2 | No image found


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

Post 46 added with 2 image(s).


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Skipping Post : 47 | No image found
Post 48 added with 1 image(s).


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Post 49 added with 4 image(s).
Post 50 added with 1 image(s).
Total post's appended:  48


# OCR Using GCP Vision

In [8]:
def is_chat_image(texts):
    # Skip if no text at all
    if len(texts) <= 1:       #checks if each image's annotated text has more than 1 line of text or not
        return False

    full_text = texts[0].description  # First item = full text
    lines = full_text.split('\n')
    if len(lines) < 20:               # in the full text checks if there are less than 4 lines then not an chat conversation
        return False

    timestamp_pattern = re.compile(r'\b(?:[01]?\d|2[0-3]):[0-5]\d(?:\s?[APap][Mm])?\b')
    timestamp_count = sum(bool(timestamp_pattern.search(line)) for line in lines)

    return timestamp_count >= 2  # You can tune this threshold


def get_image_width(image_path: str) -> int:
    with Image.open(image_path) as img:
        return img.width

# Extract blocks (with bounding box and text)
def get_blocks_from_response(response) -> List[dict]:
    blocks_out = []
    pages = response.full_text_annotation.pages

    for page in pages:
        for block in page.blocks:
            bbox = [(v.x, v.y) for v in block.bounding_box.vertices]
            text = "\n".join(
                " ".join("".join(s.text for s in w.symbols) for w in p.words)
                for p in block.paragraphs
            )
            blocks_out.append({
                "bbox": bbox,
                "text": text
            })
    return blocks_out







#function to find sort and find median using statistics package
def find_median(lst):
    if lst:
      sorted_lst = sorted(lst)

      median_value = statistics.median(sorted_lst)

      return median_value
    else:
      return None

#Check left-side block alignment consistency (scammer pattern)
def is_chat_like_by_left_alignment(blocks: List[dict], image_width: int,
                                   max_left_edge_std: int = 10,
                                   min_blocks: int = 2) -> bool:
    left_edges = []

    for b in blocks:
        xs = [x for x, _ in b["bbox"]]
        center_x = sum(xs) / len(xs)
        width = max(xs) - min(xs)   # 300 - 100 = 200

        if (image_width > 1300):
          break
        if (center_x < image_width / 2) and ( min(xs) < 200 and min(xs) > 20  ) and ((width > center_x) or (max(xs) < (image_width - image_width*0.15))):
            left_edge = min(xs)
            left_edges.append(left_edge)

    #apply median and set tolerance to remove outliers

    med_ian = find_median(left_edges)
    m = med_ian
    filtered = [x for x in left_edges if abs(x - m) <= 10]  # tolerance of 10
    # print("Left side Edges: \n")
    # print(left_edges)
    if len(left_edges) < min_blocks:
        # print("blocks kam pad gaye ")
        return False

    # left_edge_std = np.std(left_edges)
    left_edge_std = np.std(filtered)              # after applying median and tolerance
    # print("Left edge's standard deviation: ", left_edge_std)
    return left_edge_std <= max_left_edge_std

def detect_text(path):
    """Detects text in the file."""

    with open(path, 'rb') as image_file:
        content = image_file.read()
    image = vision.Image(content=content)
    response = client.text_detection(image=image)           #step 3


    texts = response.text_annotations
    # print('Texts:')
    # print(texts[0].description)

    #image width
    image_width = get_image_width(path)            #step2

    # print("Image width: ",  image_width, "\n")
    blocks = get_blocks_from_response(response)      #step 4

    # for block in blocks:            #to see all coordinates
    #     print(block)
    # print()

    is_chat = is_chat_image(texts)

    result = is_chat_like_by_left_alignment(blocks, image_width)


    if result and is_chat:
        print("✅ Likely a chat image")
        return {
            "chat": True,
            "text": texts[0].description
        }
    else:
        return {
            "chat": False,
            "text": texts[0].description if texts else ""
        }










# Saving Conversatioins to csv

In [9]:
BASE_FOLDER = "dataset_test"
CSV_FILE = "collected_data.csv"

def process_dataset():
    # check if CSV exists, else write headers
    file_exists = os.path.isfile(CSV_FILE)
    with open(CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(["post_id", "Scam_type", "Conversation"])

        # loop through scam folders
        for scam_type in os.listdir(BASE_FOLDER):
            scam_path = os.path.join(BASE_FOLDER, scam_type)
            if not os.path.isdir(scam_path):
                continue

            # loop through subfolders inside scam folder
            for post_id in os.listdir(scam_path):
                post_path = os.path.join(scam_path, post_id)
                if not os.path.isdir(post_path):
                    continue

                files = [f for f in os.listdir(post_path) if os.path.isfile(os.path.join(post_path, f))]

                # only process if subfolder has exactly one file
                if len(files) == 1:
                    image_path = os.path.join(post_path, files[0])
                    result = detect_text(image_path)
                    print(post_path)
                    # append row to csv
                    writer.writerow([
                        post_id,             # subfolder name
                        scam_type,           # scam folder name
                        result.get("text", "") if result.get("chat", False) else ""  # text only if chat
                    ])
                elif len(files) > 1:
                    combined_texts = []
                    for file in files:
                        image_path = os.path.join(post_path, file)
                        result = detect_text(image_path)
                        print(post_path)
                        if result.get("chat", False):
                            combined_texts.append(result.get("text", ""))

                    # append row to csv
                    if len(combined_texts) != 0:
                      print(combined_texts)
                      writer.writerow([
                          post_id,             # subfolder name
                          scam_type,           # scam folder name
                          " ".join(combined_texts)  # combine all texts into one string
                      ])

process_dataset()

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


dataset_test/WFH_scam/1huopb0
dataset_test/WFH_scam/1huopb0
✅ Likely a chat image
dataset_test/WFH_scam/1cxpx0m
✅ Likely a chat image
dataset_test/WFH_scam/1ljy70o
✅ Likely a chat image
dataset_test/WFH_scam/1n6ecdx
✅ Likely a chat image
dataset_test/WFH_scam/1n6ecdx
['C\n0\n1:23 PM | 0.9KB/s\n←\nPriya\n4G\n55\nlast seen recently\nPlease link net banking to work 11:54 AM\nOk but what is the work 11:55 AM ✓\nPriya\nThe job is to receive and transfer money online\nthrough banking and UPI\nThe job is to receive and transfer money\nonline through banking and UPI\n11:55 AM\nrequires you to have a current bank account\nor a business account.\nPriya\n11:55 AM\nThe job is to receive and transfer money onli...\nHow can I earn money by this?\nSalary will be sent on every Monday\n1 day you will get 1000rs and 1% of the\namount you get in your bank\nlike you get 1lak you will get 1000rs\n11:55 AM\nwhen you form a team to work for us you\nwill get a monthly bonus\n16800\n12:19 PM\nSo, I have to rec

# To Delete Empty Rows From CSV File

In [10]:
csv_file_path = "collected_data.csv"

try:
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_file_path)

    # Filter out rows where the 'Conversation' column is empty (including NaN and empty strings)
    df_filtered = df.dropna(subset=['Conversation'])
    df_filtered = df_filtered[df_filtered['Conversation'].str.strip() != '']

    # Save the filtered DataFrame back to the CSV file, overwriting the original
    df_filtered.to_csv(csv_file_path, index=False)

    print(f"Removed rows with empty 'Conversation' from '{csv_file_path}'.")
    print(f"Remaining rows: {len(df_filtered)}")

except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found.")
except KeyError:
    print(f"Error: The column 'Conversation' was not found in '{csv_file_path}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Removed rows with empty 'Conversation' from 'collected_data.csv'.
Remaining rows: 95
